In [3]:
import os

In [4]:
os.chdir("/content/drive/MyDrive/Classroom")

In [5]:
import pandas as pd

master_df = pd.read_parquet(
    "credit_risk_cleaned.parquet"
)

In [7]:
master_df.shape

(2000000, 182)

In [8]:
eda_df = master_df.sample(
    n=50000,
    random_state=42
)

(a)
Construct the following four repayment-burden features and for each report the correlation with lgd_pct:  emi_to_income_ratio  =  installment_inr ÷ (annual_inc_inr ÷ 12)  loan_to_income_ratio  =  loan_amnt_inr ÷ annual_inc_inr  rate_spread_pct  =  int_rate_pct − rbi_repo_rate_pct  real_interest_rate  =  int_rate_pct − cpi_inflation_pctWhich single feature shows the strongest correlation with the target?



In [9]:
# 1. EMI to Income Ratio = installment_inr/ (annual_inc_inr/12)
eda_df['emi_to_income_ratio'] = (
    eda_df['installment_inr']
    /
    (eda_df['annual_inc_inr'] / 12)
)

In [10]:
# Describe
eda_df['emi_to_income_ratio'].describe()

,emi_to_income_ratio
count,48963.000000
mean,0.216689
std,0.323045
min,0.000782
25%,0.052534
50%,0.115083
75%,0.249133
max,7.589533


In [11]:
# Correlation with LGD
eda_df['emi_to_income_ratio'].corr(
    eda_df['lgd_pct']
)

np.float64(0.007681079682834609)

This feature measures the proportion of monthly income required to service the loan EMI. Higher values indicate greater repayment burden and potentially increased financial stress, which may contribute to larger losses when default occurs.

In [12]:
# 2. Loan to Income Ratio = loan_annual_inr/annual_inc_inr
eda_df['loan_to_income_ratio'] = (
    eda_df['loan_amnt_inr']
    /
    eda_df['annual_inc_inr']
)

In [13]:
# Describe
eda_df['loan_to_income_ratio'].describe()

,loan_to_income_ratio
count,48963.000000
mean,0.553780
std,0.727342
min,0.002500
25%,0.148404
50%,0.314056
75%,0.656896
max,11.632120


In [14]:
# Correlation
eda_df['loan_to_income_ratio'].corr(
    eda_df['lgd_pct']
)

np.float64(0.00402662793359933)

This metric compares the loan amount with the borrower's annual income. Larger values indicate greater leverage and reduced repayment capacity relative to debt obligations.

In [15]:
# Rate Spread Percentage = int_rate-pct - rbi_repo_rate_pct
eda_df['rate_spread_feature'] = (
    eda_df['int_rate_pct']
    -
    eda_df['rbi_repo_rate_pct']
)

In [16]:
# Describe
eda_df['rate_spread_feature'].describe()

,rate_spread_feature
count,50000.000000
mean,7.424943
std,4.512382
min,-1.000000
25%,3.930000
50%,6.750000
75%,10.420000
max,23.680000


In [17]:
# Correlation
eda_df['rate_spread_feature'].corr(
    eda_df['lgd_pct']
)

np.float64(0.08557931174363859)

The rate spread measures the premium charged above the central bank policy rate. Larger spreads often reflect higher perceived borrower risk and may be associated with more severe losses.

In [18]:
# 4. Real Interest Rate = int_rate_pct - cpi_inflation_pct
eda_df['real_interest_rate_feature'] = (
    eda_df['int_rate_pct']
    -
    eda_df['cpi_inflation_pct']
)

In [19]:
# Describe
eda_df['real_interest_rate_feature'].describe()

,real_interest_rate_feature
count,50000.000000
mean,7.631782
std,4.791513
min,-3.900000
25%,4.160000
50%,7.090000
75%,10.740000
max,24.400000


In [20]:
# Correlation
eda_df['real_interest_rate_feature'].corr(
    eda_df['lgd_pct']
)

np.float64(0.07902279556059985)

Real interest rate captures the effective borrowing cost after accounting for inflation. Higher real borrowing costs may increase repayment pressure and influence credit outcomes.

In [21]:
features = [
    'emi_to_income_ratio',
    'loan_to_income_ratio',
    'rate_spread_feature',
    'real_interest_rate_feature'
]

corr_table = pd.DataFrame({
    'Feature': features,
    'Correlation_with_LGD': [
        eda_df[f].corr(eda_df['lgd_pct'])
        for f in features
    ]
})

corr_table

,Feature,Correlation_with_LGD
0,emi_to_income_ratio,0.007681
1,loan_to_income_ratio,0.004027
2,rate_spread_feature,0.085579
3,real_interest_rate_feature,0.079023


In [22]:
corr_table['Abs_Corr'] = (
    corr_table['Correlation_with_LGD']
    .abs()
)

corr_table.sort_values(
    'Abs_Corr',
    ascending=False
)

,Feature,Correlation_with_LGD,Abs_Corr
2,rate_spread_feature,0.085579,0.085579
3,real_interest_rate_feature,0.079023,0.079023
0,emi_to_income_ratio,0.007681,0.007681
1,loan_to_income_ratio,0.004027,0.004027


Strongest Correlation : -->
The rate_spread_feature exhibits the strongest correlation with lgd_pct, with a Pearson correlation coefficient of 0.0856.

Among the four engineered repayment-burden features, rate_spread_feature demonstrated the strongest relationship with LGD (r = 0.0856). However, all four features exhibit weak correlations, indicating that repayment burden metrics alone are insufficient to explain loss severity. Additional variables such as collateral coverage, recovery characteristics, and borrower credit history are likely more important determinants of LGD.

(b)
Construct the following three bureau-behaviour features:  credit_util_composite  =  0.5×revol_util + 0.3×bc_util + 0.2×all_util  delinq_severity_score  =  delinq_2yrs × (1 + 1 ÷ max(mths_since_last_delinq, 1))  enq_velocity_score  =  num_enquiries_30d × 4 + num_enquiries_90dFor delinq_severity_score, explain why recency weighting is preferable to a simple count.



In [23]:
# 1. Credit Utilization Composite
# = 0.5 * revol_util_pct + 0.3 * bs_util_pct + 0.2 * all_util_pct
eda_df['credit_util_composite'] = (
    0.5 * eda_df['revol_util_pct']
    + 0.3 * eda_df['bc_util_pct']
    + 0.2 * eda_df['all_util_pct']
)

In [24]:
eda_df['credit_util_composite'].describe()

,credit_util_composite
count,41043.000000
mean,40.630768
std,12.272088
min,4.170000
25%,31.740002
50%,40.049999
75%,49.029999
max,86.310005


In [25]:
# Correlation
eda_df['credit_util_composite'].corr(
    eda_df['lgd_pct']
)

np.float64(0.0007347057396764667)

This feature combines multiple credit utilization measures into a single indicator of revolving credit dependence. Higher utilization levels may indicate financial stress and reduced repayment flexibility.

In [26]:
# 2. Delinquency Severity Score
# = delinq_2yrs * (1+(1/max(mths_since_last_delinq,1)))
import numpy as np

eda_df['delinq_severity_score'] = (
    eda_df['delinq_2yrs']
    *
    (
        1
        +
        1 /
        np.maximum(
            eda_df['mths_since_last_delinq'],
            1
        )
    )
)

In [27]:
eda_df['delinq_severity_score'].describe()

,delinq_severity_score
count,47511.000000
mean,1.160965
std,1.766558
min,0.000000
25%,0.000000
50%,0.000000
75%,2.086957
max,14.000000


In [28]:
# Correlation
eda_df['delinq_severity_score'].corr(
    eda_df['lgd_pct']
)

np.float64(0.019820550825461837)

This feature combines both delinquency frequency and delinquency recency. Recent delinquencies are generally stronger indicators of ongoing repayment difficulties than older delinquency events.

Recency weighting is preferable because recent delinquencies provide stronger evidence of current financial distress than historical delinquencies that occurred many years ago. Two borrowers may have identical delinquency counts, but the borrower with a recent delinquency represents a higher repayment risk. Incorporating recency therefore produces a more informative measure of current credit quality.

In [29]:
# 3. Enquiry Velocity Score
# = num_enquiries_30d * 4 + num_enquiries_90d
eda_df['enq_velocity_score'] = (
    eda_df['num_enquiries_30d'] * 4
    +
    eda_df['num_enquiries_90d']
)

In [30]:
eda_df['enq_velocity_score'].describe()

,enq_velocity_score
count,50000.000000
mean,8.257560
std,7.887711
min,0.000000
25%,2.000000
50%,6.000000
75%,12.000000
max,39.000000


In [31]:
# Correlation
eda_df['enq_velocity_score'].corr(
    eda_df['lgd_pct']
)

np.float64(-0.004192888978998542)

Frequent recent credit enquiries may indicate active borrowing behavior or liquidity stress. Borrowers seeking credit from multiple lenders within a short period often exhibit elevated credit risk.

In [32]:
bureau_features = [
    'credit_util_composite',
    'delinq_severity_score',
    'enq_velocity_score'
]

corr_table_b = pd.DataFrame({
    'Feature': bureau_features,
    'Correlation_with_LGD': [
        eda_df[col].corr(eda_df['lgd_pct'])
        for col in bureau_features
    ]
})

corr_table_b['Abs_Corr'] = (
    corr_table_b['Correlation_with_LGD']
    .abs()
)

corr_table_b.sort_values(
    'Abs_Corr',
    ascending=False
)

,Feature,Correlation_with_LGD,Abs_Corr
1,delinq_severity_score,0.019821,0.019821
2,enq_velocity_score,-0.004193,0.004193
0,credit_util_composite,0.000735,0.000735


Among the three bureau-behaviour features, delinq_severity_score shows the strongest correlation with LGD, with a Pearson correlation coefficient of 0.0198.

Among the bureau-behaviour features, delinq_severity_score was the most informative predictor of LGD, although its correlation remained weak (r = 0.0198). The results suggest that bureau variables alone explain only a small portion of LGD variation and should be combined with collateral, recovery, and loan-level variables for more effective loss prediction.

(c)
Construct the following three income and collateral features:  income_stability_ratio  =  annual_inc_inr ÷ (emp_length_years + 1)  credit_depth_score  =  total_acc ÷ (credit_hist_years + 1)  collateral_coverage_ratio  =  collateral_value_inr ÷ (loan_amnt_inr + 1)Explain in business terms what a high collateral_coverage_ratio implies for Loss Given Default.



In [33]:
# 1. Income Stability Ratio
# = annual_inc_inr/(emp_length_years+1)
eda_df['income_stability_ratio'] = (
    eda_df['annual_inc_inr']
    /
    (eda_df['emp_length_years'] + 1)
)

In [34]:
eda_df['income_stability_ratio'].describe()

,income_stability_ratio
count,4.896300e+04
mean,1.431936e+05
std,2.349319e+05
min,2.777778e+03
25%,3.272333e+04
50%,7.142857e+04
75%,1.608876e+05
max,1.092659e+07


In [35]:
# Correlation
eda_df['income_stability_ratio'].corr(
    eda_df['lgd_pct']
)

np.float64(-0.002959876990964204)

This feature measures income relative to employment tenure. Borrowers with higher income and longer employment histories are generally considered more financially stable and less risky

In [36]:
# 2. Credit Depth Score
# = total_acc/(credit_hist_years+1)
eda_df['credit_depth_score'] = (
    eda_df['total_acc']
    /
    (eda_df['credit_hist_years'] + 1)
)

In [37]:
eda_df['credit_depth_score'].describe()

,credit_depth_score
count,50000.000000
mean,2.084584
std,1.967516
min,0.028653
25%,0.907369
50%,1.425965
75%,2.474271
max,16.987181


In [38]:
# Correlation
eda_df['credit_depth_score'].corr(
    eda_df['lgd_pct']
)

np.float64(-0.0007730651629622496)

This feature reflects the number of credit accounts relative to the length of the borrower’s credit history. It provides a measure of how actively a borrower uses credit over time.

In [39]:
# 3. Collateral Coverage Ratio
# = collateral_value_inr/(loan_amnt_inr+1)
eda_df['collateral_coverage_ratio'] = (
    eda_df['collateral_value_inr']
    /
    (eda_df['loan_amnt_inr'] + 1)
)

In [40]:
eda_df['collateral_coverage_ratio'].describe()

,collateral_coverage_ratio
count,50000.000000
mean,4.635415
std,12.935561
min,0.000000
25%,0.000000
50%,0.000000
75%,2.313782
max,275.867923


In [41]:
# Correlation
eda_df['collateral_coverage_ratio'].corr(
    eda_df['lgd_pct']
)

np.float64(-0.0020224064456764827)

This feature measures how much collateral value backs each rupee of loan exposure. Higher values indicate stronger security coverage for the bank.

In [42]:
income_collateral_features = [
    'income_stability_ratio',
    'credit_depth_score',
    'collateral_coverage_ratio'
]

corr_table_c = pd.DataFrame({
    'Feature': income_collateral_features,
    'Correlation_with_LGD': [
        eda_df[col].corr(eda_df['lgd_pct'])
        for col in income_collateral_features
    ]
})

corr_table_c['Abs_Corr'] = (
    corr_table_c['Correlation_with_LGD']
    .abs()
)

corr_table_c.sort_values(
    'Abs_Corr',
    ascending=False
)

,Feature,Correlation_with_LGD,Abs_Corr
0,income_stability_ratio,-0.002960,0.002960
2,collateral_coverage_ratio,-0.002022,0.002022
1,credit_depth_score,-0.000773,0.000773


A high collateral coverage ratio indicates that the value of pledged collateral is large relative to the outstanding loan amount. If the borrower defaults, the bank can recover a larger proportion of the loan through liquidation of the collateral. Consequently, higher collateral coverage is generally associated with lower Loss Given Default (LGD) and reduced credit risk exposure.

Among the three engineered income and collateral features, income_stability_ratio exhibits the strongest correlation with LGD, with a Pearson correlation coefficient of -0.0030.

All three engineered income and collateral features exhibited extremely weak correlations with LGD. Although collateral coverage ratio showed the expected negative relationship with loss severity, the magnitude was very small, indicating that LGD is influenced by additional factors such as recovery processes, borrower behavior, and loan characteristics.

(d)
Apply log(1 + x) transformation to annual_inc_inr and loan_amnt_inr to create log_annual_inc and log_loan_amnt. Report the skewness of each variable before and after transformation. Explain why reducing skewness matters for OLS regression assumptions.



In [43]:
eda_df['log_annual_inc'] = np.log1p(
    eda_df['annual_inc_inr']
)

eda_df['log_loan_amnt'] = np.log1p(
    eda_df['loan_amnt_inr']
)

In [44]:
# Skewness Table
skew_table = pd.DataFrame({
    'Variable': [
        'annual_inc_inr',
        'log_annual_inc',
        'loan_amnt_inr',
        'log_loan_amnt'
    ],
    'Skewness': [
        eda_df['annual_inc_inr'].skew(),
        eda_df['log_annual_inc'].skew(),
        eda_df['loan_amnt_inr'].skew(),
        eda_df['log_loan_amnt'].skew()
    ]
})

skew_table

,Variable,Skewness
0,annual_inc_inr,5.102005
1,log_annual_inc,0.341961
2,loan_amnt_inr,3.471409
3,log_loan_amnt,0.569323


Why does reducing skewness matter for OLS regression?
Reducing skewness improves the distributional properties of predictor variables and decreases the influence of extreme observations. Highly skewed variables can violate OLS assumptions by creating non-linearity, heteroscedasticity, and unstable coefficient estimates. Log transformation helps produce more symmetric distributions, resulting in improved model stability, more reliable parameter estimates, and better compliance with regression assumptions.

The log transformation was highly effective for both variables. Annual income skewness decreased from 5.102 to 0.342, while loan amount skewness decreased from 3.471 to 0.569. These reductions indicate that the transformed variables are considerably more suitable for regression modelling than their original forms.

(e)
Create a binary covid_issue_year_flag equal to 1 for loans issued in 2020. Present a grouped summary showing mean lgd_pct for covid_issue_year_flag = 0 vs. 1. Is the difference statistically significant? Run an independent-samples t-test and report the p-value.



1. Create covid_issue_year_flag
2. Compare mean LGD for:
0 = Non-COVID loans
1 = COVID loans (issued in 2020)

In [45]:
eda_df['covid_issue_year_flag'] = (
    eda_df['issue_year'] == 2020
).astype(int)

In [46]:
eda_df.groupby(
    'covid_issue_year_flag'
)['lgd_pct'].agg(['count','mean'])

,count,mean
covid_issue_year_flag,,
0,46522,1.577572
1,3478,1.890710


In [47]:
# T-Test
from scipy.stats import ttest_ind

covid = eda_df[
    eda_df['covid_issue_year_flag'] == 1
]['lgd_pct']

non_covid = eda_df[
    eda_df['covid_issue_year_flag'] == 0
]['lgd_pct']

t_stat, p_value = ttest_ind(
    covid,
    non_covid,
    equal_var=False
)

print("t-statistic:", t_stat)
print("p-value:", p_value)

t-statistic: 1.871854437762967
p-value: 0.06130102926115622


Loans issued during the COVID year (2020) have a higher average LGD (1.891%) compared with loans issued in non-COVID years (1.578%). The difference is approximately 0.313 percentage points, suggesting that losses were somewhat higher for loans originated during the pandemic period.

Mean LGD for non-COVID loans was 1.578%, while mean LGD for COVID-year loans was 1.891%. An independent-samples t-test produced a p-value of 0.0613, which exceeds the 0.05 significance threshold. Therefore, the difference is not statistically significant, and we fail to reject the null hypothesis that the two groups have equal mean LGD.

In [48]:
eda_df.to_parquet(
    "credit_risk_q3_features.parquet",
    index=False
)